In [ ]:
# Entrenamiento del modelo
Proyecto MIA - Detección de anomalías con Isolation Forest
import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
#Carga de datos
df = pd.read_csv("../data/sample/ejemplo_dataset.csv", sep=";")
df.head()
#Preprocesamiento basico
df["timestamp_utc"] = pd.to_datetime(df["timestamp_utc"], utc=True, errors="coerce")
df["value"] = pd.to_numeric(df["value"], errors="coerce")
df = df.dropna(subset=["timestamp_utc", "value", "instance", "metric"])

df["t15"] = df["timestamp_utc"].dt.floor("15min")

interval_df = (
    df.groupby(["t15", "instance", "metric"], as_index=False)["value"]
      .mean()
)

wide = interval_df.pivot_table(
    index=["t15", "instance"],
    columns="metric",
    values="value",
    aggfunc="mean"
).reset_index()

wide = wide.sort_values(["instance", "t15"]).reset_index(drop=True)
wide.head()

#Imputacion simple
id_cols = ["t15", "instance"]
feature_cols = [c for c in wide.columns if c not in id_cols]

for col in feature_cols:
    wide[col] = pd.to_numeric(wide[col], errors="coerce")

wide[feature_cols] = wide[feature_cols].fillna(wide[feature_cols].median(numeric_only=True))
wide.head()

#Separar variables
X = wide.drop(columns=["t15", "instance"])
X.head()
#Escalado
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

#Entrenamiento
model = IsolationForest(
    n_estimators=500,
    contamination=0.01,
    random_state=42
)

model.fit(X_scaled)

#Scores de anomalia
scores = -model.score_samples(X_scaled)
wide["anomaly_score"] = scores
wide.head()

#Umbral
threshold = np.percentile(scores, 97)
threshold
#Clasificacion
wide["is_anomaly"] = wide["anomaly_score"] >= threshold
wide[["t15", "instance", "anomaly_score", "is_anomaly"]].head()

#Conteo de anomalias
wide["is_anomaly"].value_counts()

## Resultado del entrenamiento

Se entrenó un modelo Isolation Forest sobre las métricas preprocesadas de infraestructura. Posteriormente, se calculó un puntaje de anomalía para cada observación y se definió un umbral basado en el percentil 97, con el fin de identificar comportamientos fuera del patrón normal.










